# 02 – Support Vector Machines (SVM)

SVMs find the **maximum-margin hyperplane** that separates classes.

Topics covered:
1. Linear SVM (hard & soft margin)
2. Kernel trick – RBF, Polynomial
3. SVR (Support Vector Regression)
4. Hyperparameter tuning with GridSearchCV

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import SVC, SVR
from sklearn.datasets import load_iris, make_moons, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

%matplotlib inline
np.random.seed(42)

## 1. Linear SVM – decision boundary visualisation

In [ ]:
from sklearn.datasets import make_blobs

X_lin, y_lin = make_blobs(n_samples=100, centers=2, cluster_std=1.0, random_state=42)

svm_lin = SVC(kernel='linear', C=1.0)
svm_lin.fit(X_lin, y_lin)

# Plot decision boundary
def plot_svm_boundary(clf, X, y, title='SVM Decision Boundary'):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', edgecolors='k', s=40)
    # Support vectors
    if hasattr(clf, 'support_vectors_'):
        plt.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
                    s=100, facecolors='none', edgecolors='k', linewidths=2, label='Support Vectors')
    plt.title(title); plt.legend(); plt.tight_layout(); plt.show()

plot_svm_boundary(svm_lin, X_lin, y_lin, 'Linear SVM')

## 2. RBF Kernel – non-linear data

In [ ]:
X_moon, y_moon = make_moons(n_samples=200, noise=0.2, random_state=42)

for kernel in ['linear', 'rbf', 'poly']:
    clf_k = SVC(kernel=kernel, C=1.0, gamma='scale')
    clf_k.fit(X_moon, y_moon)
    plot_svm_boundary(clf_k, X_moon, y_moon, f'SVM – kernel={kernel}')

## 3. SVM on Real Dataset (Iris) with GridSearchCV

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Pipeline: scale then SVM
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(random_state=42))
])

param_grid = {
    'svm__C':      [0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf'],
    'svm__gamma':  ['scale', 'auto']
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_tr, y_tr)

print('Best params :', grid.best_params_)
print('Best CV acc :', grid.best_score_:.3f)
print('Test acc    :', accuracy_score(y_te, grid.predict(X_te)):.3f)

## 4. Support Vector Regression (SVR)

In [ ]:
# Noisy sine wave
X_r = np.sort(np.random.uniform(0, 6, 100)).reshape(-1, 1)
y_r = np.sin(X_r).ravel() + np.random.randn(100) * 0.2

svr = SVR(kernel='rbf', C=10, epsilon=0.1)
svr.fit(X_r, y_r)

X_plot = np.linspace(0, 6, 300).reshape(-1, 1)
plt.figure(figsize=(8, 4))
plt.scatter(X_r, y_r, color='gray', s=20, label='Data')
plt.plot(X_plot, svr.predict(X_plot), color='red', label='SVR (RBF)')
plt.title('Support Vector Regression')
plt.xlabel('X'); plt.ylabel('y')
plt.legend(); plt.show()

## 5. Key Takeaways

| Concept | Notes |
|---------|-------|
| C (regularisation) | Large C → narrow margin, fits training data more tightly |
| Kernel trick | Maps to higher-dimensional space without explicit computation |
| RBF gamma | Controls influence radius; large γ → overfitting |
| SVR epsilon | Insensitive zone around the regression line |

**Next:** `03_Feature_Engineering.ipynb`